# 01 — Data Extraction
**Project:** FIFA World Cup Analytics  
**Notebook:** `01_extraction.ipynb`  
**Sector:** Sports Analytics  

## Purpose
This notebook covers the full extraction phase:
1. Document the data source and dataset metadata
2. Load the three raw CSV files into Python
3. Run a raw data quality audit — shape, dtypes, nulls, duplicates
4. Validate that all files meet the capstone minimum requirements
5. Save raw files to `data/raw/` (never to be edited)
6. Produce the data dictionary

> **Rule:** This notebook only reads and documents raw data. No transformations of any kind are applied here. All cleaning happens in `02_cleaning.ipynb`.

---
## 1. Setup

In [7]:
import pandas as pd
import numpy as np
import os
import hashlib

# Create folder structure if running for the first time
for folder in ['../data/raw', '../data/processed', '../docs']:
    os.makedirs(folder, exist_ok=True)

pd.set_option('display.max_columns', None)
print('Setup complete.')

Setup complete.


---
## 2. Data Source Documentation

| Field | Detail |
|-------|--------|
| **Source** | Kaggle — FIFA World Cup Dataset |
| **URL** | https://www.kaggle.com/datasets/abecklas/fifa-world-cup |
| **Dataset Type** | Raw / Historical transactional records |
| **Sector** | Sports Analytics |
| **Coverage** | FIFA World Cup 1930 – 2014 (20 editions) |
| **Files** | 3 CSVs — WorldCupMatches, WorldCupPlayers, WorldCups |

### Why this dataset?
- Covers 84 years of World Cup history across 3 linked tables
- Contains realistic data quality issues (blank rows, missing values, inconsistent formats)
- Supports player-level, match-level, and tournament-level analysis
- Enables KPIs across multiple dimensions: goals, attendance, team performance, player roles
- Suitable for Python ETL, statistical analysis, and Tableau dashboarding

---
## 3. Load Raw Files

Files are loaded with **no parsing, no type conversion, no filtering** — exactly as downloaded.

In [8]:
# Load all three raw files
# keep_default_na=False preserves empty strings as-is for honest null counting
matches   = pd.read_csv('../data/raw/WorldCupMatches.csv')
players   = pd.read_csv('../data/raw/WorldCupPlayers.csv')
worldcups = pd.read_csv('../data/raw/WorldCups.csv')

print('Files loaded successfully from data/raw/')
print(f'  WorldCupMatches  : {matches.shape[0]:>6,} rows  x  {matches.shape[1]} columns')
print(f'  WorldCupPlayers  : {players.shape[0]:>6,} rows  x  {players.shape[1]} columns')
print(f'  WorldCups        : {worldcups.shape[0]:>6,} rows  x  {worldcups.shape[1]} columns')

Files loaded successfully from data/raw/
  WorldCupMatches  :  4,572 rows  x  20 columns
  WorldCupPlayers  : 37,784 rows  x  9 columns
  WorldCups        :     20 rows  x  10 columns


---
## 4. Raw Data Audit

We document the exact state of each file before any cleaning — this is the baseline record for the project report.

In [9]:
def raw_audit(df, name):
    """Print a full raw quality report for a DataFrame."""
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Rows         : {df.shape[0]:,}')
    print(f'  Columns      : {df.shape[1]}')
    print(f'  Duplicates   : {df.duplicated().sum():,}')
    print(f'  Total nulls  : {df.isnull().sum().sum():,}')
    print(f'  Null %       : {df.isnull().mean().mean()*100:.1f}%')
    print(f'  Memory (KB)  : {df.memory_usage(deep=True).sum()/1024:.1f}')
    print()
    print('  Columns, Dtypes, and Null Counts:')
    summary = pd.DataFrame({
        'dtype'     : df.dtypes,
        'null_count': df.isnull().sum(),
        'null_%'    : (df.isnull().mean() * 100).round(1),
        'unique'    : df.nunique()
    })
    print(summary.to_string())
    print()
    print('  First 3 rows:')
    print(df.head(3).to_string())

In [10]:
raw_audit(matches, 'WorldCupMatches.csv')


  WorldCupMatches.csv
  Rows         : 4,572
  Columns      : 20
  Duplicates   : 3,735
  Total nulls  : 74,402
  Null %       : 81.4%
  Memory (KB)  : 2293.1

  Columns, Dtypes, and Null Counts:
                        dtype  null_count  null_%  unique
Year                  float64        3720    81.4      20
Datetime               object        3720    81.4     602
Stage                  object        3720    81.4      23
Stadium                object        3720    81.4     181
City                   object        3720    81.4     151
Home Team Name         object        3720    81.4      78
Home Team Goals       float64        3720    81.4      11
Away Team Goals       float64        3720    81.4       7
Away Team Name         object        3720    81.4      83
Win conditions         object        3720    81.4      43
Attendance            float64        3722    81.4     622
Half-time Home Goals  float64        3720    81.4       7
Half-time Away Goals  float64        3720    81.4

In [11]:
raw_audit(players, 'WorldCupPlayers.csv')


  WorldCupPlayers.csv
  Rows         : 37,784
  Columns      : 9
  Duplicates   : 736
  Total nulls  : 62,356
  Null %       : 18.3%
  Memory (KB)  : 12114.9

  Columns, Dtypes, and Null Counts:
                dtype  null_count  null_%  unique
RoundID         int64           0     0.0     101
MatchID         int64           0     0.0     836
Team Initials  object           0     0.0      82
Coach Name     object           0     0.0     335
Line-up        object           0     0.0       2
Shirt Number    int64           0     0.0      24
Player Name    object           0     0.0    7663
Position       object       33641    89.0       3
Event          object       28715    76.0    1893

  First 3 rows:
   RoundID  MatchID Team Initials           Coach Name Line-up  Shirt Number       Player Name Position Event
0      201     1096           FRA  CAUDRON Raoul (FRA)       S             0       Alex THEPOT       GK   NaN
1      201     1096           MEX     LUQUE Juan (MEX)       S     

In [12]:
raw_audit(worldcups, 'WorldCups.csv')


  WorldCups.csv
  Rows         : 20
  Columns      : 10
  Duplicates   : 0
  Total nulls  : 0
  Null %       : 0.0%
  Memory (KB)  : 7.4

  Columns, Dtypes, and Null Counts:
                 dtype  null_count  null_%  unique
Year             int64           0     0.0      20
Country         object           0     0.0      15
Winner          object           0     0.0       9
Runners-Up      object           0     0.0      10
Third           object           0     0.0      14
Fourth          object           0     0.0      16
GoalsScored      int64           0     0.0      17
QualifiedTeams   int64           0     0.0       5
MatchesPlayed    int64           0     0.0       9
Attendance      object           0     0.0      20

  First 3 rows:
   Year  Country   Winner      Runners-Up    Third      Fourth  GoalsScored  QualifiedTeams  MatchesPlayed Attendance
0  1930  Uruguay  Uruguay       Argentina      USA  Yugoslavia           70              13             18    590.549
1  1934    

---
## 5. Issues Log — What Was Found in Raw Data

This section records every quality issue identified during the audit. These become the cleaning instructions for `02_cleaning.ipynb`.

In [13]:
issues = pd.DataFrame({
    'File': [
        'WorldCupMatches', 'WorldCupMatches', 'WorldCupMatches',
        'WorldCupMatches', 'WorldCupMatches', 'WorldCupMatches',
        'WorldCupPlayers', 'WorldCupPlayers', 'WorldCupPlayers',
        'WorldCupPlayers', 'WorldCupPlayers',
        'WorldCups'
    ],
    'Column': [
        'All columns', 'All columns', 'Year / RoundID / MatchID',
        'Datetime', 'Attendance', 'Win conditions',
        'All rows', 'Position', 'Event',
        'Shirt Number', 'Line-up',
        'Attendance'
    ],
    'Issue': [
        '3,720 rows where every column is NaN (padding rows)',
        '3,735 duplicate rows',
        'Stored as float64 due to NaN rows — should be integer',
        'Plain string, not a datetime type',
        '2 missing values + stored as float',
        'Empty string for regular-time wins — not filterable',
        '736 duplicate rows',
        '33,641 nulls — position not recorded in early tournaments',
        '28,715 nulls — players with no goal/card/sub have no event',
        'Value 0 used as placeholder for unknown shirt numbers',
        'Codes S/N — opaque, needs readable labels',
        'Stored as string "590.549" (European thousands separator)'
    ],
    'Planned Fix': [
        'dropna(how=all)',
        'drop_duplicates()',
        'Cast to int after blank row removal',
        'pd.to_datetime() with format string',
        'Fill with year-level median, cast to int',
        'Replace empty string with Normal',
        'drop_duplicates()',
        'fillna(Unknown)',
        'fillna(No Event)',
        'Replace 0 with NaN',
        'Map S to Starting, N to Substitute',
        'Remove dot separator, cast to int'
    ]
})

print(issues.to_string(index=False))

           File                   Column                                                      Issue                              Planned Fix
WorldCupMatches              All columns        3,720 rows where every column is NaN (padding rows)                          dropna(how=all)
WorldCupMatches              All columns                                       3,735 duplicate rows                        drop_duplicates()
WorldCupMatches Year / RoundID / MatchID      Stored as float64 due to NaN rows — should be integer      Cast to int after blank row removal
WorldCupMatches                 Datetime                          Plain string, not a datetime type      pd.to_datetime() with format string
WorldCupMatches               Attendance                         2 missing values + stored as float Fill with year-level median, cast to int
WorldCupMatches           Win conditions        Empty string for regular-time wins — not filterable         Replace empty string with Normal
WorldCupPlaye

---
## 6. Capstone Requirements Validation

Verify this dataset meets the minimum standards defined in the handbook.

In [14]:
print('===== CAPSTONE REQUIREMENTS CHECK =====')
print()

# Requirement 1: Minimum 5,000 rows
total_rows = matches.shape[0] + players.shape[0] + worldcups.shape[0]
row_check = total_rows >= 5000
print(f'[{"PASS" if row_check else "FAIL"}] Minimum 5,000 rows')
print(f'         Total rows across all files: {total_rows:,}')
print(f'         WorldCupPlayers alone: {players.shape[0]:,} rows')

print()

# Requirement 2: Minimum 8 meaningful analytical columns
# After merging, the master dataset will have 37 columns
# WorldCupPlayers alone has 9 columns, WorldCupMatches has 20
col_check = players.shape[1] >= 8
print(f'[{"PASS" if col_check else "FAIL"}] Minimum 8 meaningful analytical columns')
print(f'         WorldCupMatches  : {matches.shape[1]} columns')
print(f'         WorldCupPlayers  : {players.shape[1]} columns')
print(f'         WorldCups        : {worldcups.shape[1]} columns')
print(f'         Merged (estimated): ~37 columns')

print()

# Requirement 3: Contains realistic data quality issues
has_nulls = players.isnull().sum().sum() > 0
has_dupes = players.duplicated().sum() > 0
has_type_issues = matches['Year'].dtype == float
issues_check = has_nulls and has_dupes and has_type_issues
print(f'[{"PASS" if issues_check else "FAIL"}] Contains realistic data quality issues')
print(f'         Missing values present : {has_nulls} (Position: 33,641 nulls)')
print(f'         Duplicates present     : {has_dupes} (Players: 736 dupes)')
print(f'         Type issues present    : {has_type_issues} (Year stored as float)')

print()

# Requirement 4: Tabular row-level records
print(f'[PASS] Tabular row-level records (CSV format, suitable for Pandas + Tableau)')

print()
all_pass = row_check and col_check and issues_check
print(f'OVERALL: {"ALL CHECKS PASSED — Dataset approved for capstone" if all_pass else "SOME CHECKS FAILED"}')

===== CAPSTONE REQUIREMENTS CHECK =====

[PASS] Minimum 5,000 rows
         Total rows across all files: 42,376
         WorldCupPlayers alone: 37,784 rows

[PASS] Minimum 8 meaningful analytical columns
         WorldCupMatches  : 20 columns
         WorldCupPlayers  : 9 columns
         WorldCups        : 10 columns
         Merged (estimated): ~37 columns

[PASS] Contains realistic data quality issues
         Missing values present : True (Position: 33,641 nulls)
         Duplicates present     : True (Players: 736 dupes)
         Type issues present    : True (Year stored as float)

[PASS] Tabular row-level records (CSV format, suitable for Pandas + Tableau)

OVERALL: ALL CHECKS PASSED — Dataset approved for capstone


---
## 7. Relationship Between the Three Files

Understanding how the files connect is critical before any merge in the cleaning notebook.

In [15]:
print('===== FILE RELATIONSHIP ANALYSIS =====')
print()

# Check join key: MatchID overlap between Matches and Players
matches_clean_ids = set(matches.dropna(subset=['MatchID'])['MatchID'].astype(int))
players_ids       = set(players['MatchID'].unique())

print('Join Key 1: RoundID + MatchID  (WorldCupPlayers ↔ WorldCupMatches)')
print(f'  Unique MatchIDs in WorldCupMatches : {len(matches_clean_ids)}')
print(f'  Unique MatchIDs in WorldCupPlayers : {len(players_ids)}')
print(f'  Overlapping MatchIDs               : {len(matches_clean_ids & players_ids)}')
print(f'  Join Type to use                   : LEFT JOIN (keep all player records)')

print()

# Check join key: Year overlap between Matches and WorldCups
match_years = set(matches.dropna(subset=['Year'])['Year'].astype(int).unique())
cup_years   = set(worldcups['Year'].unique())

print('Join Key 2: Year  (WorldCupMatches ↔ WorldCups)')
print(f'  Unique Years in WorldCupMatches : {sorted(match_years)}')
print(f'  Unique Years in WorldCups       : {sorted(cup_years)}')
print(f'  Year overlap                    : {len(match_years & cup_years)} / {len(match_years)}')
print(f'  Join Type to use                : LEFT JOIN (keep all match records)')

print()
print('Merge Strategy for 02_cleaning.ipynb:')
print('  Step 1: WorldCupPlayers LEFT JOIN WorldCupMatches  ON [RoundID, MatchID]')
print('  Step 2: Result          LEFT JOIN WorldCups        ON Year')
print('  Output: world_cup_master.csv  (~37,048 rows x 37 columns)')
print('  Grain:  One row = One player in one match in one tournament')

===== FILE RELATIONSHIP ANALYSIS =====

Join Key 1: RoundID + MatchID  (WorldCupPlayers ↔ WorldCupMatches)
  Unique MatchIDs in WorldCupMatches : 836
  Unique MatchIDs in WorldCupPlayers : 836
  Overlapping MatchIDs               : 836
  Join Type to use                   : LEFT JOIN (keep all player records)

Join Key 2: Year  (WorldCupMatches ↔ WorldCups)
  Unique Years in WorldCupMatches : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014)]
  Unique Years in WorldCups       : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int

---
## 8. Data Dictionary

Field-level documentation for all three raw files. This feeds directly into `docs/data_dictionary.md`.

In [16]:
print('=== WorldCupMatches.csv ===')

dict_matches = pd.DataFrame({
    'Column': [
        'Year','Datetime','Stage','Stadium','City',
        'Home Team Name','Home Team Goals','Away Team Goals','Away Team Name',
        'Win conditions','Attendance',
        'Half-time Home Goals','Half-time Away Goals',
        'Referee','Assistant 1','Assistant 2',
        'RoundID','MatchID','Home Team Initials','Away Team Initials'
    ],
    'Raw Type': [
        'float64','object','object','object','object',
        'object','float64','float64','object',
        'object','float64',
        'float64','float64',
        'object','object','object',
        'float64','float64','object','object'
    ],
    'Description': [
        'Year of the World Cup tournament',
        'Match date and kick-off time (string format)',
        'Tournament stage (Group, Quarter-Final, etc.)',
        'Name of the stadium where match was played',
        'City where match was played',
        'Full name of home team',
        'Goals scored by home team (full time)',
        'Goals scored by away team (full time)',
        'Full name of away team',
        'Special win condition: Extra Time, Penalties (empty = Normal)',
        'Number of spectators at the match',
        'Home team goals at half time',
        'Away team goals at half time',
        'Referee name and nationality',
        'Assistant referee 1 name and nationality',
        'Assistant referee 2 name and nationality',
        'Round identifier (join key with WorldCupPlayers)',
        'Unique match identifier (join key with WorldCupPlayers)',
        '3-letter initials of home team',
        '3-letter initials of away team'
    ],
    'Known Issues': [
        'float due to 3,720 blank rows — cast to int after drop',
        'String not datetime — parse with pd.to_datetime()',
        'Clean','Clean','Trailing whitespace in some values',
        'Trailing whitespace','float — cast to int','float — cast to int','Clean',
        'Empty string for regular wins — fill with Normal',
        '2 nulls — fill with year median',
        'float — cast to int','float — cast to int',
        'Clean','Drop — not analytically useful','Drop — not analytically useful',
        'float — cast to int','float — cast to int','Clean','Clean'
    ]
})
print(dict_matches.to_string(index=False))

=== WorldCupMatches.csv ===
              Column Raw Type                                                   Description                                           Known Issues
                Year  float64                              Year of the World Cup tournament float due to 3,720 blank rows — cast to int after drop
            Datetime   object                  Match date and kick-off time (string format)      String not datetime — parse with pd.to_datetime()
               Stage   object                 Tournament stage (Group, Quarter-Final, etc.)                                                  Clean
             Stadium   object                    Name of the stadium where match was played                                                  Clean
                City   object                                   City where match was played                     Trailing whitespace in some values
      Home Team Name   object                                        Full name of home tea

In [17]:
print('=== WorldCupPlayers.csv ===')

dict_players = pd.DataFrame({
    'Column': [
        'RoundID','MatchID','Team Initials','Coach Name',
        'Line-up','Shirt Number','Player Name','Position','Event'
    ],
    'Raw Type': [
        'int64','int64','object','object',
        'object','int64','object','object','object'
    ],
    'Description': [
        'Round identifier (join key with WorldCupMatches)',
        'Unique match identifier (join key with WorldCupMatches)',
        '3-letter team code for the player',
        'Coach name and nationality in brackets',
        'S = Starting XI, N = Named substitute',
        'Player jersey number (0 = unknown)',
        'Player full name',
        'Playing position: GK, DF, MF, FW',
        'Match event: goal (G40), yellow card (Y65), substitution (SU78)'
    ],
    'Known Issues': [
        'Clean — int, ready to join',
        'Clean — int, ready to join',
        'Clean',
        'Clean',
        'Opaque codes — decode S/N to Starting/Substitute',
        '0 used as placeholder for unknown — replace with NaN',
        'Clean',
        '33,641 nulls — not recorded in early tournaments — fill Unknown',
        '28,715 nulls — no event = no action in that match — fill No Event'
    ]
})
print(dict_players.to_string(index=False))

=== WorldCupPlayers.csv ===
       Column Raw Type                                                     Description                                                      Known Issues
      RoundID    int64                Round identifier (join key with WorldCupMatches)                                        Clean — int, ready to join
      MatchID    int64         Unique match identifier (join key with WorldCupMatches)                                        Clean — int, ready to join
Team Initials   object                               3-letter team code for the player                                                             Clean
   Coach Name   object                          Coach name and nationality in brackets                                                             Clean
      Line-up   object                           S = Starting XI, N = Named substitute                  Opaque codes — decode S/N to Starting/Substitute
 Shirt Number    int64                              Pl

In [18]:
print('=== WorldCups.csv ===')

dict_cups = pd.DataFrame({
    'Column': [
        'Year','Country','Winner','Runners-Up','Third','Fourth',
        'GoalsScored','QualifiedTeams','MatchesPlayed','Attendance'
    ],
    'Raw Type': [
        'int64','object','object','object','object','object',
        'int64','int64','int64','object'
    ],
    'Description': [
        'Year of the tournament (join key with other files)',
        'Country that hosted the tournament',
        'Country that won the tournament',
        'Country that finished as runner-up',
        'Country that finished third',
        'Country that finished fourth',
        'Total goals scored across all matches in the tournament',
        'Number of national teams that qualified',
        'Total number of matches played',
        'Total spectators across all tournament matches'
    ],
    'Known Issues': [
        'Clean — int, ready to join',
        'Rename to host_country to avoid ambiguity',
        'Clean','Clean','Clean','Clean',
        'Clean','Clean','Clean',
        'String using European dot separator (590.549 = 590,549) — convert to int'
    ]
})
print(dict_cups.to_string(index=False))

=== WorldCups.csv ===
        Column Raw Type                                             Description                                                             Known Issues
          Year    int64      Year of the tournament (join key with other files)                                               Clean — int, ready to join
       Country   object                      Country that hosted the tournament                                Rename to host_country to avoid ambiguity
        Winner   object                         Country that won the tournament                                                                    Clean
    Runners-Up   object                      Country that finished as runner-up                                                                    Clean
         Third   object                             Country that finished third                                                                    Clean
        Fourth   object                            Country t

---
## 9. File Integrity Check

Generate MD5 checksums of the raw files. This creates a tamper-proof record proving the raw files were never modified after extraction.

In [19]:
def md5_checksum(filepath):
    """Return MD5 hash of a file for integrity verification."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

raw_files = [
    '../data/raw/WorldCupMatches.csv',
    '../data/raw/WorldCupPlayers.csv',
    '../data/raw/WorldCups.csv'
]

print('Raw File Integrity Checksums (MD5):')
print(f'{"File":<35} {"MD5 Checksum"}')
print('-' * 75)
for f in raw_files:
    try:
        chk = md5_checksum(f)
        size = os.path.getsize(f) / 1024
        print(f'{os.path.basename(f):<35} {chk}  ({size:.1f} KB)')
    except FileNotFoundError:
        print(f'{os.path.basename(f):<35} FILE NOT FOUND — place in data/raw/')

print()
print('Save these checksums in docs/data_dictionary.md as proof the raw files were never edited.')

Raw File Integrity Checksums (MD5):
File                                MD5 Checksum
---------------------------------------------------------------------------
WorldCupMatches.csv                 fcf8786e01cabb0c0e2fd0f7f89f1944  (233.4 KB)
WorldCupPlayers.csv                 51f2a1c3355cd146e142850ddfef3585  (2100.2 KB)
WorldCups.csv                       53eb0d9d731d780e4114807c54ce6947  (1.4 KB)

Save these checksums in docs/data_dictionary.md as proof the raw files were never edited.


---
## 10. Extraction Summary

| Item | Detail |
|------|--------|
| Files extracted | 3 CSV files |
| Total raw rows | 42,376 (4,572 + 37,784 + 20) |
| Total raw columns | 39 (20 + 9 + 10) |
| Join keys identified | RoundID + MatchID (Players ↔ Matches) · Year (Matches ↔ Cups) |
| Merge strategy | Players LEFT JOIN Matches LEFT JOIN WorldCups |
| Estimated clean rows | ~37,048 (after blank/dup removal) |
| Estimated clean columns | 37 (merged + 3 derived) |
| Data quality issues found | 12 (documented in Issues Log above) |
| Raw files committed to | `data/raw/` — never to be edited |
| Next step | `02_cleaning.ipynb` — clean, merge, and export to `data/processed/` |